# Stage 2: Preprocessing (KO)

**This stage is diagnostic only: it produces no artifacts that downstream stages depend on.** It can be skipped on re-runs.

Its purpose is to sanity-check three things before Stages 3-7 commit expensive compute:

1. **Tokenization fits within budget.** All four NLP models are tokenized against the headline corpus to verify that `CONFIG["nlp"]["max_length"]` (currently 128) is large enough that essentially no headlines are truncated.
2. **Shared chronological cutoff is well-defined.** Stages 3-5 all rely on `get_shared_chronological_cutoff` returning a single cutoff date that produces a sensible train/test split for both text and numerical data; this stage prints that boundary so it can be eyeballed.
3. **Both modalities split on the same date.** A bug where the cutoff fell on different dates for text vs. numerical data would silently invalidate every cross-modal comparison in Stages 4-7.

No random sampling and no artifacts are written. If everything looks reasonable here, the rest of the pipeline can be trusted to produce comparable text and numerical evaluations.


In [1]:
# Shared config, paths, and helpers.
from common import *

# Tokenizer class for token-length inspection per model.
from transformers import AutoTokenizer

[common] device=cpu  artifacts=/cluster/tufts/hrilab/jmonta04/modular_pipeline/artifacts  results=/cluster/tufts/hrilab/jmonta04/modular_pipeline/results


## 2.1 Load artifacts from Stage 1

In [2]:
# Load Stage 1 artifacts.
text_df = load_text_df()
num_df = load_num_df()

# Date-only values so all downstream joins/splits align cleanly.
text_df["date"] = pd.to_datetime(text_df["date"]).dt.normalize()
num_df["date"] = pd.to_datetime(num_df["date"]).dt.normalize()

# One shared cutoff date so text and numerical test sets cover the same
# out-of-sample period.
cutoff_date = get_shared_chronological_cutoff(
    text_df=text_df,
    num_df=num_df,
    test_size=CONFIG["regression"]["test_size"],
)

# Sanity checks on loaded data size and split point.
print(f"text_df rows: {len(text_df)}")
print(f"num_df rows:  {len(num_df)}")
print(f"Shared chronological cutoff: {cutoff_date.date()}")

text_df rows: 10521
num_df rows:  3356
Shared chronological cutoff: 2021-06-10


## 2.2 Tokenization statistics per NLP model

In [3]:
# Each NLP model uses its own tokenizer (different vocabularies, different
# subword splits). The same headline produces different token counts under
# DistilBERT vs. BERT-base vs. RoBERTa vs. FinBERT, so all four must be
# inspected. The number that matters most is "Num truncated at 128": if any
# tokenizer truncates more than a small handful of headlines, signal would
# be lost at training time and CONFIG["nlp"]["max_length"] should be bumped.
# KO headlines (typically short news titles) produce a truncation count near 0.
for model, path in NLP_MODELS.items():
    # Build tokenizer for the current checkpoint.
    tok = AutoTokenizer.from_pretrained(path)

    # Raw token length per headline without truncation.
    lengths = text_df["text"].apply(
        lambda text: len(tok(text, truncation=False, padding=False)["input_ids"])
    )

    # Training-time token cap from config.
    max_len = CONFIG["nlp"]["max_length"]
    # Rows that would be truncated at training time.
    n_trunc = int((lengths > max_len).sum())

    print(f"Model: {model}")
    print(f"  Num truncated at {max_len}: {n_trunc}")
    print(f"  Min/median/max tokens: {lengths.min()} / {lengths.median():.1f} / {lengths.max()}")
    print("  Percentiles (50/90/95/99/100):")
    print(lengths.quantile([0.5, 0.9, 0.95, 0.99, 1.0]).to_string())
    print()

Model: DistilBERT
  Num truncated at 128: 14
  Min/median/max tokens: 6 / 28.0 / 172
  Percentiles (50/90/95/99/100):
0.50     28.0
0.90     46.0
0.95     60.0
0.99     94.0
1.00    172.0



Model: BERT-base
  Num truncated at 128: 14
  Min/median/max tokens: 6 / 28.0 / 172
  Percentiles (50/90/95/99/100):
0.50     28.0
0.90     46.0
0.95     60.0
0.99     94.0
1.00    172.0



Model: RoBERTa-base
  Num truncated at 128: 15
  Min/median/max tokens: 8 / 32.0 / 176
  Percentiles (50/90/95/99/100):
0.50     32.0
0.90     52.0
0.95     64.0
0.99     96.0
1.00    176.0



Model: FinBERT
  Num truncated at 128: 14
  Min/median/max tokens: 6 / 28.0 / 172
  Percentiles (50/90/95/99/100):
0.50     28.0
0.90     46.0
0.95     60.0
0.99     94.0
1.00    172.0



## 2.3 Chronological split sanity checks

Validates that both text and numerical data use the same train/test cutoff date.


In [4]:
# Confirms that the same cutoff date produces consistent train/test boundaries
# for both modalities. Both `text_train_df["date"].max()` and
# `num_train_df["date"].max()` should equal `cutoff_date`, and the test mins
# should be the next available date in each table. If these print otherwise,
# Stages 3 and 4 would be evaluating on misaligned out-of-sample periods.

# Chronological split for text data (train = early dates, test = later dates).
text_train_df, text_test_df = chronological_split_by_date(
    df=text_df,
    date_col="date",
    cutoff_date=cutoff_date,
)

# Chronological split + scaling for numerical data.
(
    X_train,
    X_test,
    y_train,
    y_test,
    _,
    feature_cols,
    num_train_df,
    num_test_df,
) = make_num_splits_chronological(
    df=num_df,
    cutoff_date=cutoff_date,
    target_col="risk_score",
    date_col="date",
)

# Show split boundaries to confirm no future leakage.
print("Text split:")
print(f"  train rows: {len(text_train_df)}")
print(f"  test rows:  {len(text_test_df)}")
print(f"  train max date: {text_train_df['date'].max().date()}")
print(f"  test min date:  {text_test_df['date'].min().date()}")

# Numerical matrix shapes and date boundaries.
print("\nNumerical split:")
print(f"  X_train shape: {X_train.shape}")
print(f"  X_test shape:  {X_test.shape}")
print(f"  feature count: {len(feature_cols)}")
print(f"  train max date: {num_train_df['date'].max().date()}")
print(f"  test min date:  {num_test_df['date'].min().date()}")

Text split:
  train rows: 7975
  test rows:  2546
  train max date: 2021-06-10
  test min date:  2021-06-11

Numerical split:
  X_train shape: (2734, 11)
  X_test shape:  (622, 11)
  feature count: 11
  train max date: 2021-06-10
  test min date:  2021-06-11
